## Aeropulse — Silver Transformation: Exploration

**Purpose:** Scratch / profiling notebook — not part of the production bronze-to-silver run chain. Used to inspect bronze `airport`, `carrier` and `flight` data (row counts, null checks, grain checks, value encodings, whitespace issues) ahead of designing the transformation logic that the three `*-bronze-to-silver` notebooks implement.

**Reads:** `aeropulse__bronze__lh.dbo.bronze_airport`, `bronze_carrier`, `bronze_flight` (via SparkSQL temp views)

**Default lakehouse:** `aeropulse__bronze__lh`

**Note:** Findings from this notebook (data quality flags, key-column grain, rename decisions) are already reflected in the three production silver notebooks — keep this one out of any scheduled/production run.


In [1]:

df = spark.sql("SELECT * FROM aeropulse__bronze__lh.dbo.bronze_airport")
##display(df.limit(100))

StatementMeta(, 4510fe0e-8f20-44e6-a668-56c68699fbd9, 3, Finished, Available, Finished, False)

In [2]:
# create temp view for SparkSQL querying
df.createOrReplaceTempView('airport_view')

StatementMeta(, 4510fe0e-8f20-44e6-a668-56c68699fbd9, 4, Finished, Available, Finished, False)

In [6]:
%%sql

--SELECT * FROM airport_view LIMIT 10;

-- check count of rows
--ELECT COUNT(*) FROM airport_view
-- 6,818 rows

-- check distinct count of column, 'code'
--SELECT COUNT(DISTINCT code) as dst_code_count FROM airport_view;
-- 6818

-- check null values in column. 'code'
--SELECT COUNT(*) FROM airport_view WHERE code IS NULL;
-- outcome: 0 - no null in this column

-- check how values in column, description is stored
--SELECT * FROM airport_view LIMIT 100;
-- values are stored in this form: Bensalem, PA: Total Rf Heliport"- three attributes can be derived from this: airport_name, airport_state, airport_city

-- comfirm if this value pattern "Bensalem, PA: Total Rf Heliport" holds across dataset
--SELECT COUNT(*) FROM airport_view WHERE Description NOT LIKE '%, __: %';
-- there are 3814 with different pattern

--SELECT *
--FROM(
   -- SELECT * FROM airport_view WHERE Description NOT LIKE '%, _ _: %'
--)
--LIMIT 100
--outcome takes similar shape: airport_city_state : airport_name

-- check if there are null values in column, description
--SELECT COUNT(*) FROM airport_view WHERE Description IS NULL;
-- outcome: 0 - no null value in this column

-- check for leading/trailing whitespace on code and Description
SELECT COUNT(*) as whitepace_issues FROM airport_view WHERE code <> TRIM(code) OR Description <> TRIM(Description);
-- outcome: 0 - 

-- final: 1. rename column headers: all in small and snake case
        --2. rename column to descriptive naming convention
        --3. derive the following column from description(renamed to airport_description:): airport_state, airport_city, airport_name
        --4. add a data quality flag - drop future entries with code with null
        --5. generate a surrogate key column




StatementMeta(, 4510fe0e-8f20-44e6-a668-56c68699fbd9, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## exploring carrier data
 - understand the data qualtity status
 - count of rows
 - check for nulls on important attributes: primary/unique key
 - understand how column headers/column values are been stored
 

In [1]:
carrier_df = spark.sql("SELECT * FROM aeropulse__bronze__lh.dbo.bronze_carrier")
## display(carrier_df)

StatementMeta(, 5578ae26-9bca-4483-b4a1-7265f25511e6, 3, Finished, Available, Finished, False)

# creating a carrier temp view for SparkSql querying

In [2]:
carrier_df.createOrReplaceTempView('carrier_view')

StatementMeta(, 5578ae26-9bca-4483-b4a1-7265f25511e6, 4, Finished, Available, Finished, False)

In [3]:
%%sql

-- count number of rows
--SELECT COUNT(*) FROM carrier_view;
-- 1,751

-- check the distinct count of column, code
--SELECT * FROM carrier_view LIMIT 100;
--SELECT COUNT(DISTINCT code) as dst_cnt_code FROM carrier_view;
-- code: 1751 distinct count

-- check null values in column, code
--SELECT COUNT(*) FROM carrier_view WHERE code IS NULL;
-- outcome: 0 - no null values in this column

-- check for null in column, description
--SELECT COUNT(*) FROM carrier_view WHERE Description IS NULL;
-- outcome: 0 - no null values in this column

-- check for leading/trailing whitespace on code and Description
SELECT COUNT(*) as whitepace_issues FROM carrier_view WHERE code <> TRIM(code) OR Description <> TRIM(Description);
-- outcome: 0 - 


-- final: 1. rename column headers: all in small and snake case
        --2. rename column to descriptive naming convention
        --3. add a data quality flag - drop future entries with code with null
        --4. generate a surrogate key column



StatementMeta(, 5578ae26-9bca-4483-b4a1-7265f25511e6, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## exploring flight data
 - understand the data qualtity status
 - count of rows
 - check for nulls on important attributes: primary/unique key
 - understand how column headers/column values are been stored
 

In [1]:
flight_df = spark.sql("SELECT * FROM aeropulse__bronze__lh.dbo.bronze_flight")
##display(flight_df)

StatementMeta(, 9bc1832f-87e2-4677-9962-1c8c22182ca7, 3, Finished, Available, Finished, False)

## create a temp flight_view for SparkSql querying

In [2]:
flight_df.createOrReplaceTempView('flight_view')

StatementMeta(, 9bc1832f-87e2-4677-9962-1c8c22182ca7, 4, Finished, Available, Finished, False)

In [4]:
%%sql

--SELECT * FROM flight_view LIMIT 100;

-- number of rows
--SELECT COUNT(*) FROM flight_view;
-- 570,138 rows

-- check for nulls in key attributes:
--SELECT COUNT(*) as null_check FROM flight_view WHERE FL_DATE IS NULL OR OP_UNIQUE_CARRIER IS NULL OR OP_CARRIER_FL_NUM IS NULL OR ORIGIN IS NULL OR DEST IS NULL;
-- outcome: 0

--check grain: does FL_DATE + OP_UNIQUE_CARRIER + OP_CARRIER_FL_NUM + ORIGIN uniquely identify a row?
--SELECT COUNT(*) as total_rows, COUNT(DISTINCT FL_DATE, OP_UNIQUE_CARRIER, OP_CARRIER_FL_NUM, ORIGIN) as distinct_grain FROM flight_view;
-- outcome: total_rows = distinct grain

-- check distinct values of cancelled and diverted to understand how these flags are encoded
--SELECT DISTINCT CANCELLED FROM flight_view;
-- value stored as 1 and 0 - map values boolean
--SELECT DISTINCT DIVERTED FROM flight_view;
-- value stored as 1 and 0 - map values as boolean

-- data quality: cancelled flights should have no actual departure/arrival times
--SELECT COUNT(*) as cancelled_with_time FROM flight_view WHERE CAST(CANCELLED AS DOUBLE) = 1 AND (DEP_TIME IS NOT NULL OR ARR_TIME IS NOT NULL)
-- outcome: 345 occurence where a cancelled flight has either arrival time or departure time

-- data quality: non-cancelled flights should have a null cancellation code
--SELECT COUNT(*) as non_cancelled_with_code FROM flight_view WHERE CAST(CANCELLED AS DOUBLE) = 0 and CANCELLATION_CODE IS NOT NULL;
-- outcome: 0

-- data quality: check for negative delay values
--SELECT COUNT(*) as negative_dep_delay FROM flight_view WHERE CAST(DEP_DELAY AS DOUBLE) < 0;
-- OUTCOME: 343,390 occurence
--SELECT COUNT(*) as negative_dep_delay FROM flight_view WHERE CAST(ARR_DELAY AS DOUBLE) < 0;
-- outcome: 362,491

-- check how often each delay-cause column is populated (expected to be null whenever a flight is not delayed)
--SELECT COUNT(*) as total, COUNT(CARRIER_DELAY) as carrier_delay_populated, COUNT(WEATHER_DELAY) as weather_delay_populated, COUNT(NAS_DELAY) as nas_delay_populated,
--COUNT(SECURITY_DELAY) as security_delay_populated, COUNT(LATE_AIRCRAFT_DELAY) as late_aircraft_delay_populated FROM flight_view;
--outcome carrier_delay_populated = weather_delay_populated = nas_delay_populated = 97,760

SELECT COUNT(*) as total,
       COUNT(SECURITY_DELAY) as security_delay_populated,
       COUNT(LATE_AIRCRAFT_DELAY) as late_aircraft_delay_populated
FROM flight_view;

-- Final: 1. rename column headers: all lowercase and snake case format
        --2. cast/delay/time/numeric columns (DEP_DELAY, ARR_DELAY, DISTANCE, AIR_TIME, and the individual delay-cause columns) from string to numeric types
        --3. cast FL_DATE to a proper date column
        --4. create a FL_DATE_STRING
        --5. standardise CANCELLED/DIVERTED to boolean
        --6. add a data quality flag for cancelled-with-times and non-cancelled-with-code for inconsistency check
        --7. generate a surrogate key column

StatementMeta(, 9bc1832f-87e2-4677-9962-1c8c22182ca7, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>